# NER Aplicado a Negocios
## Reconocimiento de Entidades Nombradas con spaCy

**Objetivo:** Aprender a extraer información estructurada de textos empresariales usando NER.

**Casos de negocio que cubriremos:**
1. Análisis de noticias financieras (detectar empresas, montos, fechas)
2. Procesamiento de facturas y documentos comerciales
3. Monitoreo de reputación de marca en redes sociales
4. Extracción de información de correos empresariales

---

## 1. Instalación y configuración del entorno

Primero instalamos spaCy y el modelo en español. Este paso solo se ejecuta una vez.

In [2]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================================
# spaCy: librería de PLN industrial
# es_core_news_md: modelo en español (mediano, incluye word vectors)

!pip install -q spacy pandas
!python -m spacy download es_core_news_md -q
!python -m spacy download en_core_web_sm -q

print("Instalación completada OK")

✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Instalación completada OK


## 2. Carga del modelo y primer ejemplo

Cargamos el modelo de spaCy y verificamos los componentes del pipeline.

In [3]:
# ============================================================
# CARGA DEL MODELO Y EXPLORACIÓN DEL PIPELINE
# ============================================================

import spacy

# Cargar modelo en español (mediano)
nlp = spacy.load("es_core_news_md")

# Ver los componentes del pipeline de procesamiento
# Cada componente transforma el texto en una etapa diferente
print("Componentes del pipeline:")
for name, component in nlp.pipeline:
    print(f"  {name:20s} -> {type(component).__name__}")

print(f"\nTotal de componentes: {len(nlp.pipeline)}")
print(f"Vocabulario: {len(nlp.vocab)} palabras")

Componentes del pipeline:
  tok2vec              -> Tok2Vec
  morphologizer        -> Morphologizer
  parser               -> DependencyParser
  attribute_ruler      -> AttributeRuler
  lemmatizer           -> SpanishLemmatizer
  ner                  -> EntityRecognizer

Total de componentes: 6
Vocabulario: 420 palabras


## 3. Caso de Negocio: Análisis de Noticias Financieras

**Escenario:** El departamento de inteligencia de mercado necesita monitorear noticias
sobre competidores, inversiones y movimientos del mercado. NER automatiza la extracción
de datos clave de cientos de artículos diarios.

---

In [4]:
# ============================================================
# CASO 1: EXTRACCIÓN DE ENTIDADES EN NOTICIAS FINANCIERAS
# ============================================================

import spacy

nlp = spacy.load("es_core_news_md")

# Simulamos noticias financieras reales de un feed empresarial
noticias_financieras = [
    "Bancolombia reportó utilidades netas por $3.2 billones en el tercer trimestre de 2024, un aumento del 15% respecto al mismo periodo del año anterior.",
    "Ecopetrol firmó un acuerdo de exploración conjunta con Petrobrás en el Golfo de México por un valor estimado de USD 500 millones el 12 de noviembre.",
    "El Grupo Éxito anunció el cierre de 15 tiendas en Bogotá y Medellín como parte de su plan de reestructuración para 2025.",
    "La Superintendencia Financiera de Colombia impuso una multa de $800 millones a Davivienda por irregularidades en el reporte de operaciones sospechosas.",
    "Amazon Web Services inauguró su primera región de nube en Colombia, con una inversión de USD 2.000 millones en infraestructura tecnológica.",
]

# Procesar cada noticia y extraer entidades
print("=" * 80)
print("ANÁLISIS DE NOTICIAS FINANCIERAS")
print("=" * 80)

for i, noticia in enumerate(noticias_financieras):
    doc = nlp(noticia)
    print(f"\n📰 Noticia {i+1}: {noticia[:80]}...")
    print("-" * 60)

    if doc.ents:
        for ent in doc.ents:
            # Mostrar cada entidad con su tipo y posición en el texto
            print(f"  🏷️  {ent.text:30s} | Tipo: {ent.label_:8s} | Pos: {ent.start_char}-{ent.end_char}")
    else:
        print("  ⚠️  No se detectaron entidades")

ANÁLISIS DE NOTICIAS FINANCIERAS

📰 Noticia 1: Bancolombia reportó utilidades netas por $3.2 billones en el tercer trimestre de...
------------------------------------------------------------
  🏷️  Bancolombia                    | Tipo: LOC      | Pos: 0-11

📰 Noticia 2: Ecopetrol firmó un acuerdo de exploración conjunta con Petrobrás en el Golfo de ...
------------------------------------------------------------
  🏷️  Ecopetrol                      | Tipo: ORG      | Pos: 0-9
  🏷️  Petrobrás                      | Tipo: PER      | Pos: 55-64
  🏷️  Golfo de México                | Tipo: LOC      | Pos: 71-86

📰 Noticia 3: El Grupo Éxito anunció el cierre de 15 tiendas en Bogotá y Medellín como parte d...
------------------------------------------------------------
  🏷️  Grupo Éxito                    | Tipo: MISC     | Pos: 3-14
  🏷️  Bogotá                         | Tipo: LOC      | Pos: 50-56
  🏷️  Medellín                       | Tipo: LOC      | Pos: 59-67

📰 Noticia 4: La Superint

### 3.1 Construir un DataFrame estructurado desde las noticias

Transformamos la información extraída en una tabla que puede alimentar dashboards,
bases de datos o sistemas de Business Intelligence.

In [5]:
# ============================================================
# PIPELINE: NOTICIAS → NER → DATAFRAME ESTRUCTURADO
# ============================================================

import pandas as pd

# Función reutilizable para extraer entidades de cualquier lista de textos
def extraer_entidades(textos, nlp_model, fuente="desconocida"):
    """
    Pipeline de extracción de entidades.

    Args:
        textos: lista de strings con textos a procesar
        nlp_model: modelo de spaCy cargado
        fuente: nombre de la fuente de datos

    Returns:
        DataFrame con las entidades extraídas
    """
    registros = []

    # nlp.pipe() procesa en lote (más eficiente que un bucle con nlp())
    for idx, doc in enumerate(nlp_model.pipe(textos, batch_size=50)):
        for ent in doc.ents:
            registros.append({
                "texto_id": idx,
                "fuente": fuente,
                "entidad": ent.text,
                "tipo": ent.label_,
                "inicio": ent.start_char,
                "fin": ent.end_char,
                "contexto": doc.text[max(0, ent.start_char-20):ent.end_char+20]
            })

    return pd.DataFrame(registros)

# Ejecutar el pipeline sobre las noticias financieras
df_entidades = extraer_entidades(noticias_financieras, nlp, fuente="noticias_financieras")

# Mostrar resultados
print(f"Total de entidades extraídas: {len(df_entidades)}")
print(f"\nDistribución por tipo:")
print(df_entidades["tipo"].value_counts().to_string())
print(f"\nPrimeras 10 entidades:")
df_entidades.head(10)

Total de entidades extraídas: 11

Distribución por tipo:
tipo
LOC     5
MISC    3
ORG     2
PER     1

Primeras 10 entidades:


,texto_id,fuente,entidad,tipo,inicio,fin,contexto
0,0,noticias_financieras,Bancolombia,LOC,0,11,Bancolombia reportó utilidades
1,1,noticias_financieras,Ecopetrol,ORG,0,9,Ecopetrol firmó un acuerdo de
2,1,noticias_financieras,Petrobrás,PER,55,64,ración conjunta con Petrobrás en el Golfo de Méxi
3,1,noticias_financieras,Golfo de México,LOC,71,86,con Petrobrás en el Golfo de México por un val...
4,2,noticias_financieras,Grupo Éxito,MISC,3,14,El Grupo Éxito anunció el cierre d
5,2,noticias_financieras,Bogotá,LOC,50,56,re de 15 tiendas en Bogotá y Medellín como par
6,2,noticias_financieras,Medellín,LOC,59,67,tiendas en Bogotá y Medellín como parte de su pl
7,3,noticias_financieras,Superintendencia Financiera de Colombia,ORG,3,42,La Superintendencia Financiera de Colombia imp...
8,3,noticias_financieras,Davivienda,MISC,79,89,de $800 millones a Davivienda por irregularid...
9,4,noticias_financieras,Amazon Web Services,MISC,0,19,Amazon Web Services inauguró su primera


## 4. Caso de Negocio: Procesamiento de Correos Empresariales

**Escenario:** El equipo de ventas recibe cientos de correos diarios. NER puede extraer
automáticamente los contactos, empresas, fechas de reunión y montos mencionados,
alimentando el CRM de forma automática.

---

In [6]:
# ============================================================
# CASO 2: EXTRACCIÓN DE DATOS DE CORREOS EMPRESARIALES
# ============================================================

correos = [
    """Estimado equipo, les confirmo que la reunión con el gerente de Alpina,
    Carlos Rodríguez, se realizará el próximo 15 de enero en las oficinas de Bogotá.
    El presupuesto aprobado para el proyecto es de $450 millones.""",

    """Buenos días, informo que Nestlé Colombia ha aceptado nuestra propuesta
    comercial por USD 120.000 anuales. El contrato será firmado por María González,
    directora de compras, antes del 28 de febrero de 2025.""",

    """Atención: el cliente Cementos Argos solicitó una extensión del plazo
    de entrega hasta el 30 de marzo. El representante legal, Andrés Mejía,
    enviará la solicitud formal a nuestra sede en Medellín.""",
]

# Procesar correos
df_correos = extraer_entidades(correos, nlp, fuente="correos_ventas")

print("📧 ENTIDADES EXTRAÍDAS DE CORREOS EMPRESARIALES")
print("=" * 70)

# Agrupar por tipo para facilitar el análisis
for tipo in df_correos["tipo"].unique():
    subset = df_correos[df_correos["tipo"] == tipo]
    print(f"\n🏷️  {tipo} ({len(subset)} encontradas):")
    for _, row in subset.iterrows():
        print(f"    • {row['entidad']}")

# Mostrar tabla completa
print("\n\n📋 Tabla completa:")
df_correos[["texto_id", "entidad", "tipo", "fuente"]]

📧 ENTIDADES EXTRAÍDAS DE CORREOS EMPRESARIALES

🏷️  LOC (5 encontradas):
    • Alpina
    • Bogotá
    • Buenos días
    • Nestlé Colombia
    • Medellín

🏷️  PER (3 encontradas):
    • Carlos Rodríguez
    • María González
    • Andrés Mejía

🏷️  MISC (3 encontradas):
    • El contrato
    • Cementos Argos
    • El representante legal


📋 Tabla completa:


,texto_id,entidad,tipo,fuente
0,0,Alpina,LOC,correos_ventas
1,0,Carlos Rodríguez,PER,correos_ventas
2,0,Bogotá,LOC,correos_ventas
3,1,Buenos días,LOC,correos_ventas
4,1,Nestlé Colombia,LOC,correos_ventas
5,1,El contrato,MISC,correos_ventas
6,1,María González,PER,correos_ventas
7,2,Cementos Argos,MISC,correos_ventas
8,2,El representante legal,MISC,correos_ventas
9,2,Andrés Mejía,PER,correos_ventas


## 5. Caso de Negocio: Monitoreo de Marca en Redes Sociales

**Escenario:** El equipo de marketing necesita monitorear menciones de la marca
y competidores en redes sociales para detectar tendencias y crisis reputacionales.

---

In [7]:
# ============================================================
# CASO 3: MONITOREO DE MARCA EN REDES SOCIALES
# ============================================================

# Simulamos tweets/publicaciones sobre marcas
publicaciones = [
    "Terrible servicio de Rappi en Bogotá, llevo 2 horas esperando mi pedido del Éxito.",
    "Increíble la nueva app de Bancolombia, transferencias en 5 segundos 🚀",
    "Comparando precios: Amazon tiene mejor oferta que Mercado Libre para tecnología en Colombia.",
    "El CEO de Nubank, David Vélez, anunció expansión a 5 ciudades colombianas en enero 2025.",
    "Mal servicio postventa de Samsung Colombia, llevan 3 semanas sin resolver mi caso.",
    "Avianca canceló mi vuelo Bogotá-Cartagena sin previo aviso. Pérdida de $2 millones.",
]

# Extraer entidades
df_social = extraer_entidades(publicaciones, nlp, fuente="redes_sociales")

# Filtrar solo organizaciones mencionadas (marcas/empresas)
marcas = df_social[df_social["tipo"] == "ORG"]

print("🏢 MARCAS MENCIONADAS EN REDES SOCIALES")
print("=" * 50)
if len(marcas) > 0:
    print(marcas["entidad"].value_counts().to_string())
else:
    print("No se detectaron organizaciones")

# Filtrar personas mencionadas
personas = df_social[df_social["tipo"] == "PER"]
print(f"\n👤 PERSONAS MENCIONADAS: {len(personas)}")
for _, row in personas.iterrows():
    print(f"   • {row['entidad']}")

# Filtrar lugares
lugares = df_social[df_social["tipo"] == "LOC"]
print(f"\n📍 LUGARES MENCIONADOS: {len(lugares)}")
for _, row in lugares.iterrows():
    print(f"   • {row['entidad']}")

🏢 MARCAS MENCIONADAS EN REDES SOCIALES
entidad
Rappi               1
Mercado Libre       1
Samsung Colombia    1
Avianca             1

👤 PERSONAS MENCIONADAS: 3
   • Nubank
   • David Vélez
   • Bogotá-Cartagena

📍 LUGARES MENCIONADOS: 4
   • Bogotá
   • Increíble
   • Bancolombia
   • Colombia


## 6. Pipeline avanzado: Matcher para detectar relaciones comerciales

Usamos el Matcher de spaCy para detectar patrones específicos como
"[EMPRESA] firmó contrato con [EMPRESA]" o "[PERSONA] es gerente de [EMPRESA]".

---

In [8]:
# ============================================================
# MATCHER: DETECTAR RELACIONES COMERCIALES EN TEXTOS
# ============================================================

from spacy.matcher import Matcher

nlp = spacy.load("es_core_news_md")
matcher = Matcher(nlp.vocab)

# Patrón 1: Detectar acuerdos comerciales
# Busca: [EMPRESA/ORG] + verbo de acuerdo + ... + [EMPRESA/ORG]
patron_acuerdo = [
    {"ENT_TYPE": "ORG", "OP": "+"},
    {"LEMMA": {"IN": ["firmar", "acordar", "negociar", "cerrar", "anunciar"]}, "OP": "+"},
    {"OP": "*"},  # tokens intermedios
    {"ENT_TYPE": "ORG", "OP": "+"},
]
matcher.add("ACUERDO_COMERCIAL", [patron_acuerdo])

# Textos de ejemplo empresarial
textos_negocios = [
    "Bancolombia firmó un acuerdo estratégico con Microsoft para modernizar su infraestructura digital.",
    "Ecopetrol negoció condiciones especiales con Schlumberger para los pozos del Meta.",
    "El Grupo Éxito anunció una alianza con Rappi para expandir su servicio de entregas.",
]

print("🤝 RELACIONES COMERCIALES DETECTADAS")
print("=" * 60)

for texto in textos_negocios:
    doc = nlp(texto)
    matches = matcher(doc)

    print(f"\n📄 Texto: {texto[:70]}...")
    if matches:
        for match_id, start, end in matches:
            span = doc[start:end]
            patron_nombre = nlp.vocab.strings[match_id]
            print(f"   ✅ Patrón: {patron_nombre}")
            print(f"      Fragmento: \"{span.text}\"")

            # Extraer las entidades dentro del match
            ents_en_match = [ent for ent in doc.ents if ent.start >= start and ent.end <= end]
            for ent in ents_en_match:
                print(f"      → {ent.label_}: {ent.text}")
    else:
        print("   ❌ No se encontró patrón de acuerdo")

🤝 RELACIONES COMERCIALES DETECTADAS

📄 Texto: Bancolombia firmó un acuerdo estratégico con Microsoft para modernizar...
   ❌ No se encontró patrón de acuerdo

📄 Texto: Ecopetrol negoció condiciones especiales con Schlumberger para los poz...
   ✅ Patrón: ACUERDO_COMERCIAL
      Fragmento: "Ecopetrol negoció condiciones especiales con Schlumberger"
      → ORG: Ecopetrol
      → ORG: Schlumberger

📄 Texto: El Grupo Éxito anunció una alianza con Rappi para expandir su servicio...
   ❌ No se encontró patrón de acuerdo


## 7. Dependencias sintácticas para extraer tripletas de negocio

Extraemos relaciones sujeto-verbo-objeto para entender quién hizo qué en contextos empresariales.

---

In [9]:
# ============================================================
# TRIPLETAS SUJETO-VERBO-OBJETO EN CONTEXTO EMPRESARIAL
# ============================================================

nlp = spacy.load("es_core_news_md")

textos_empresariales = [
    "Bancolombia adquirió el 40% de las acciones de Nequi.",
    "La junta directiva aprobó el presupuesto para 2025.",
    "El gerente comercial presentó la propuesta a los inversionistas.",
]

def extraer_tripletas(doc):
    """Extrae tripletas (sujeto, verbo, objeto) de un documento procesado."""
    tripletas = []
    for token in doc:
        # Buscar verbos (raíz de la oración)
        if token.pos_ == "VERB":
            sujetos = [child.text for child in token.children
                       if child.dep_ in ("nsubj", "nsubj:pass")]
            objetos = [child.text for child in token.children
                       if child.dep_ in ("obj", "dobj", "obl", "iobj")]

            if sujetos or objetos:
                tripletas.append({
                    "sujeto": " | ".join(sujetos) if sujetos else "[no detectado]",
                    "verbo": token.lemma_,   # forma base del verbo
                    "objeto": " | ".join(objetos) if objetos else "[no detectado]",
                })
    return tripletas

print("🔗 TRIPLETAS EXTRAÍDAS DE TEXTOS EMPRESARIALES")
print("=" * 60)

for texto in textos_empresariales:
    doc = nlp(texto)
    tripletas = extraer_tripletas(doc)

    print(f"\n📄 \"{texto}\"")
    for t in tripletas:
        print(f"   ({t['sujeto']}) --[{t['verbo']}]--> ({t['objeto']})")

    # Mostrar árbol de dependencias completo
    print("   Dependencias:")
    for token in doc:
        if token.dep_ != "punct":
            print(f"      {token.text:20s} dep={token.dep_:12s} head={token.head.text}")

🔗 TRIPLETAS EXTRAÍDAS DE TEXTOS EMPRESARIALES

📄 "Bancolombia adquirió el 40% de las acciones de Nequi."
   (Bancolombia) --[adquirir]--> (40%)
   Dependencias:
      Bancolombia          dep=nsubj        head=adquirió
      adquirió             dep=ROOT         head=adquirió
      el                   dep=det          head=40%
      40%                  dep=obj          head=adquirió
      de                   dep=case         head=acciones
      las                  dep=det          head=acciones
      acciones             dep=nmod         head=40%
      de                   dep=case         head=Nequi
      Nequi                dep=nmod         head=acciones

📄 "La junta directiva aprobó el presupuesto para 2025."
   (junta) --[aprobar]--> (presupuesto)
   Dependencias:
      La                   dep=det          head=junta
      junta                dep=nsubj        head=aprobó
      directiva            dep=amod         head=junta
      aprobó               dep=ROOT         head=a

## 8. Visualización con displaCy

displaCy permite visualizar las entidades detectadas directamente en Jupyter.

---

In [10]:
# ============================================================
# VISUALIZACIÓN INTERACTIVA CON DISPLACY
# ============================================================

from spacy import displacy

# Texto de ejemplo empresarial
texto = """Bancolombia reportó utilidades de $3.2 billones en el tercer
trimestre de 2024. El presidente Juan Carlos Mora confirmó que la inversión
en tecnología alcanzará los USD 500 millones para 2025 en Colombia."""

doc = nlp(texto)

# Visualizar entidades (en Jupyter se renderiza automáticamente)
# style="ent" resalta las entidades con colores
displacy.render(doc, style="ent", jupyter=True)

In [11]:
# ============================================================
# VISUALIZACIÓN DE DEPENDENCIAS SINTÁCTICAS
# ============================================================

# Texto corto para ver las relaciones gramaticales
texto_corto = "Bancolombia adquirió Nequi en 2024."
doc_corto = nlp(texto_corto)

# style="dep" muestra las relaciones entre palabras
displacy.render(doc_corto, style="dep", jupyter=True,
                options={"distance": 120, "compact": True})

## Resumen y conclusiones

### Lo que aprendimos:
- **NER** transforma texto libre en datos estructurados (DataFrames)
- **nlp.pipe()** permite procesamiento eficiente en lote
- **Matcher** detecta patrones complejos (relaciones comerciales)
- **Dependencias sintácticas** extraen tripletas sujeto-verbo-objeto
- **displaCy** visualiza entidades y relaciones

### Aplicaciones de negocio cubiertas:
1. ✅ Análisis de noticias financieras
2. ✅ Procesamiento de correos empresariales
3. ✅ Monitoreo de marca en redes sociales
4. ✅ Detección de relaciones comerciales
5. ✅ Extracción de tripletas de conocimiento
